<a href="https://colab.research.google.com/github/NgoMinhHai-arch/OmniSuite/blob/main/Ollama_Setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Run Ollama in Colab
---

[![5aharsh/collama](https://raw.githubusercontent.com/5aharsh/collama/main/assets/banner.png)](https://github.com/5aharsh/collama)

This is an example notebook which demonstrates how to run Ollama inside a Colab instance. With this you can run pretty much any small to medium sized models offerred by Ollama for free.

For the list of available models check [models being offerred by Ollama](https://ollama.com/library).


## Before you proceed
---

Since by default the runtime type of Colab instance is CPU based, in order to use LLM models make sure to change your runtime type to T4 GPU (or better if you're a paid Colab user). This can be done by going to **Runtime > Change runtime type**.

While running your script be mindful of the resources you're using. This can be tracked at **Runtime > View resources**.

## Running the notebook
---

After configuring the runtime just run it with **Runtime > Run all**. And you can start tinkering around. This example uses [Llama 3.2](https://ollama.com/library/llama3.2) to generate a response from a prompted question using [LangChain Ollama Integration](https://python.langchain.com/docs/integrations/chat/ollama/).

## Installing Dependencies
---

1. `pciutils` is required by Ollama to detect the GPU type.
2. Installation of Ollama in the runtime instance will be taken care by `curl -fsSL https://ollama.com/install.sh | sh`




In [5]:
!sudo apt update
!sudo apt install -y pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
132 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as r

## Running Ollama
---

In order to use Ollama it needs to run as a service in background parallel to your scripts. Becasue Jupyter Notebooks is built to run code blocks in sequence this make it difficult to run two blocks at the same time. As a workaround we will create a service using subprocess in Python so it doesn't block any cell from running.

Service can be started by command `ollama serve`.

`time.sleep(5)` adds some delay to get the Ollama service up before downloading the model.

In [6]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

## Pulling Model
---

Download the LLM model using `ollama pull llama3.2`.

For other models check https://ollama.com/library

In [7]:
!ollama pull mistral-nemo

## And that's it!
---

With this you should be able to freely play around with the models in your scripts. Following is an example using `langchain-ollama` to answer a simple prompt.

If you have a use-case that can help out others feel free to add your notebook to [Collama](https://github.com/5aharsh/collama/fork)

In [9]:
!pip install langchain-ollama

In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM
from IPython.display import Markdown

template = """Question: {question}

Answer: Let's think step by step."""

prompt = ChatPromptTemplate.from_template(template)

# Tôi đã đổi tên model thành mistral-nemo cho bạn
model = OllamaLLM(model="mistral-nemo")

chain = prompt | model

# Bạn có thể thay đổi câu hỏi trong ngoặc kép dưới đây
display(Markdown(chain.invoke({"question": "Chào bạn, bạn có thể giới thiệu về bản thân mình bằng tiếng Việt không?"})))

ConnectError: [Errno 111] Connection refused

In [13]:
import requests
import time
import subprocess
import threading

def start_ollama():
    # Kiểm tra xem Ollama có đang chạy không
    try:
        requests.get("http://localhost:11434/api/tags")
        print("Ollama đã sẵn sàng.")
    except:
        print("Đang khởi động lại Ollama...")
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        # Đợi cho đến khi API phản hồi
        for _ in range(10):
            try:
                requests.get("http://localhost:11434/api/tags")
                print("Ollama đã khởi động thành công!")
                return
            except:
                time.sleep(2)
        print("Không thể kết nối với Ollama. Vui lòng kiểm tra lại bước cài đặt.")

start_ollama()

# Chạy lại inference
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM
from IPython.display import Markdown

template = """Question: {question}

Answer: Let's think step by step."""
prompt = ChatPromptTemplate.from_template(template)
model = OllamaLLM(model="mistral-nemo")
chain = prompt | model

print("Đang gửi câu hỏi đến Mistral Nemo...")
display(Markdown(chain.invoke({"question": "Chào bạn, bạn có thể giới thiệu về bản thân mình bằng tiếng Việt không?"})))

Đang khởi động lại Ollama...
Ollama đã khởi động thành công!
Đang gửi câu hỏi đến Mistral Nemo...


Chào bạn! Tôi là một mô hình ngôn ngữ được tạo bởi nhân vật AI. Tôi không có thông tin cá nhân hoặc lịch sử của riêng tôi. Tuy nhiên, tôi có thể nói với bạn rằng tôi được thiết kế để giúp đỡ trả lời câu hỏi và cung cấp thông tin trong nhiều lĩnh vực khác nhau, bao gồm cả tiếng Việt. Xin vui lòng đặt câu hỏi nếu bạn cần sự hỗ trợ!